In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Mitigasi ralat rangkaian tensor (TEM): Fungsi Qiskit oleh Algorithmiq
> **Note:** Qiskit Functions adalah ciri eksperimental yang hanya tersedia untuk pengguna Plan Premium IBM Quantum&reg;, Plan Flex, dan Plan On-Prem (melalui IBM Quantum Platform API). Ia berada dalam status keluaran pratonton dan tertakluk kepada perubahan.


<details>
<summary><b>Versi pakej</b></summary>

Kod pada halaman ini dibangunkan menggunakan keperluan berikut.
Kami mengesyorkan menggunakan versi ini atau yang lebih baharu.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Gambaran keseluruhan
Kaedah Mitigasi Ralat Rangkaian Tensor (TEM) Algorithmiq adalah algoritma kuantum-klasikal hibrid yang direka untuk melakukan mitigasi hingar sepenuhnya pada peringkat pasca-pemprosesan klasikal. Dengan TEM, pengguna boleh mengira nilai jangkaan pemerhatian yang mengurangkan ralat yang disebabkan oleh hingar yang tidak dapat dielakkan pada perkakasan kuantum dengan ketepatan dan kecekapan kos yang ditingkatkan, menjadikannya pilihan yang sangat menarik bagi penyelidik kuantum dan pengamal industri.

Kaedah ini terdiri daripada membina rangkaian tensor yang mewakili songsangan saluran hingar global yang mempengaruhi keadaan pemproses kuantum dan kemudian menggunakan peta tersebut pada hasil pengukuran lengkap maklumat yang diperoleh daripada keadaan bising untuk mendapatkan penganggar tidak berat sebelah bagi pemerhatian.

Sebagai kelebihan, TEM memanfaatkan pengukuran lengkap maklumat untuk memberi akses kepada set nilai jangkaan pemerhatian yang dikurangkan yang luas dan mempunyai overhead persampelan yang optimum pada perkakasan kuantum, seperti yang diterangkan dalam Filippov et al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), dan Filippov et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542). Overhead pengukuran merujuk kepada bilangan pengukuran tambahan yang diperlukan untuk melakukan mitigasi ralat yang cekap, faktor kritikal dalam kebolehlaksanaan pengiraan kuantum. Oleh itu, TEM berpotensi membolehkan kelebihan kuantum dalam senario kompleks, seperti aplikasi dalam bidang huru-hara kuantum, fizik banyak badan, dinamik Hubbard, dan simulasi kimia molekul kecil.

Ciri-ciri dan manfaat utama TEM boleh diringkaskan sebagai:

1.  **Overhead pengukuran optimum**: TEM adalah optimum berkenaan
[had teori](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
bermakna tiada kaedah yang boleh mencapai overhead pengukuran yang lebih kecil. Dalam
erti kata lain, TEM memerlukan bilangan minimum pengukuran tambahan untuk melakukan
mitigasi ralat. Ini seterusnya bermakna TEM menggunakan masa runtime kuantum yang minimum.
2.  **Kos efektif**: Memandangkan TEM mengendalikan mitigasi hingar sepenuhnya dalam peringkat pasca-pemprosesan, tidak perlu menambah litar tambahan kepada komputer kuantum, yang bukan sahaja menjadikan pengiraan lebih murah tetapi juga mengurangkan risiko memperkenalkan ralat tambahan akibat ketidaksempurnaan peranti kuantum.
3.  **Anggaran pelbagai pemerhatian**: Berkat pengukuran lengkap maklumat, TEM menganggarkan pelbagai pemerhatian dengan cekap menggunakan data pengukuran yang sama daripada komputer kuantum.
4.  **Mitigasi ralat pengukuran**: Fungsi TEM Qiskit juga termasuk
[kaedah mitigasi ralat pengukuran proprietari](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154)
yang mampu mengurangkan ralat bacaan dengan ketara selepas jalankan kalibrasi yang singkat.
5.  **Ketepatan**: TEM meningkatkan ketepatan dan kebolehpercayaan simulasi kuantum digital dengan ketara, menjadikan algoritma kuantum lebih tepat dan boleh dipercayai.
## Keterangan
Fungsi TEM membolehkan anda mendapatkan nilai jangkaan yang dikurangkan ralat untuk pelbagai pemerhatian pada Circuit kuantum dengan overhead persampelan yang minimum. Circuit diukur dengan ukuran operator bernilai pengendali positif yang lengkap maklumat (IC-POVM), dan hasil pengukuran yang dikumpulkan diproses pada komputer klasikal. Pengukuran ini digunakan untuk melakukan kaedah rangkaian tensor dan membina peta songsangan hingar. Fungsi ini menggunakan peta yang menyongsangkan sepenuhnya keseluruhan Circuit bising menggunakan rangkaian tensor untuk mewakili lapisan bising.

![TEM schematics](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Anggaran pemerhatian O yang dikurangkan ralat melalui hasil pengukuran pasca-pemprosesan pemproses kuantum bising. U dan N menandakan operasi kuantum ideal dan peta hingar yang berkaitan, yang secara amnya boleh bukan tempatan (dan diperluas kepada kotak kelabu). D mewakili tensor pengendali yang dual kepada kesan dalam pengukuran IC. Modul mitigasi hingar M adalah rangkaian tensor yang dikontrak dengan cekap dari tengah ke luar. Iterasi pertama penguncupan diwakili oleh garis ungu bertitik, yang kedua oleh garis putus-putus, dan yang ketiga oleh garis pepejal.")

Sebaik sahaja litar diserahkan ke fungsi, ia ditranspilasikan dan dioptimumkan untuk meminimumkan bilangan lapisan dengan get dua Qubit (get yang lebih bising pada peranti kuantum). Hingar yang mempengaruhi lapisan dipelajari melalui
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
menggunakan model hingar Pauli-Lindblad jarang seperti yang diterangkan dalam E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023).
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Model hingar adalah keterangan tepat hingar pada peranti yang mampu menangkap ciri-ciri halus, termasuk cross-talk Qubit. Walau bagaimanapun, hingar pada peranti boleh turun naik dan hanyut dan hingar yang dipelajari mungkin tidak tepat pada ketika anggaran dibuat. Ini boleh mengakibatkan keputusan yang tidak tepat.
## Mulakan
Sahkan menggunakan [kunci API IBM Quantum Platform](http://quantum.cloud.ibm.com/) anda, dan pilih fungsi TEM seperti berikut. (Coretan ini mengandaikan anda sudah [menyimpan akaun anda](/guides/functions#install-qiskit-functions-catalog-client) ke persekitaran tempatan anda.)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Contoh
Coretan berikut menunjukkan contoh di mana TEM digunakan untuk mengira nilai jangkaan pemerhatian untuk Circuit kuantum mudah.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Gunakan API Qiskit Serverless untuk memeriksa status beban kerja Fungsi Qiskit anda:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Anda boleh mengembalikan keputusan seperti:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Nilai jangkaan untuk Circuit tanpa hingar bagi pengendali yang diberikan sepatutnya sekitar `0.18409094298943401`.
## Input
**Parameter**

Nama | Jenis | Keterangan | Diperlukan | Lalai | Contoh
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Iterable objek seperti PUB (blok bersatu primitif), seperti tupel `(circuit, observables)` atau `(circuit, observables, parameters, precision)`. Lihat [Gambaran keseluruhan PUB](/guides/primitive-input-output#overview-of-pubs) untuk maklumat lanjut. Jika Circuit bukan-ISA dihantar, ia akan ditranspilasikan dengan tetapan optimum. Jika Circuit ISA dihantar, ia tidak akan ditranspilasikan; dalam kes ini, pemerhatian mesti ditakrifkan pada keseluruhan QPU. | Ya | T/A | (circuit, observables)
`backend_name` | str | Nama Backend untuk membuat pertanyaan.| Tidak | Jika tidak disediakan, Backend paling kurang sibuk akan digunakan. | "ibm_fez"
`options` | dict | Pilihan input. Lihat bahagian `Options` untuk maklumat lanjut. | Tidak | Lihat bahagian `Options` untuk maklumat lanjut.| {"max_bond_dimension": 100\}

> **Caution:** TEM pada masa ini mempunyai had berikut:
> 
>   - Litar berparameter tidak disokong. Argumen parameter perlu ditetapkan kepada `None` jika ketepatan dinyatakan. Sekatan ini akan dibuang dalam versi masa hadapan.
>   - Hanya litar tanpa gelung yang disokong. Sekatan ini akan dibuang dalam versi masa hadapan.
>   - Get bukan-unitari, seperti reset, ukur, dan semua bentuk aliran kawalan tidak disokong. Sokongan untuk reset akan ditambahkan dalam keluaran akan datang.
### Pilihan
Kamus yang mengandungi pilihan lanjutan untuk TEM. Kamus ini boleh mengandungi kunci dalam jadual berikut. Jika mana-mana pilihan tidak disediakan, nilai lalai yang disenaraikan dalam jadual akan digunakan. Nilai lalai adalah baik untuk penggunaan tipikal TEM.

Nama | Pilihan | Keterangan  | Lalai
-- | -- | -- | --
`tem_max_bond_dimension` | int | Dimensi ikatan maksimum yang hendak digunakan untuk rangkaian tensor. | 500 |
`tem_compression_cutoff` | float | Nilai potongan yang hendak digunakan untuk rangkaian tensor. | 1e-16
`compute_shadows_bias_from_observable` | bool | Bendera boolean yang menunjukkan sama ada berat sebelah untuk protokol pengukuran bayangan klasikal perlu disesuaikan dengan pemerhatian PUB atau tidak. Jika False, protokol bayangan klasikal (kebarangkalian sama mengukur Z, X, Y) akan digunakan.| False
`shadows_bias` | np.ndarray | Berat sebelah yang hendak digunakan untuk protokol pengukuran bayangan klasikal rawak, tatasusunan 1d atau 2d bersaiz 3 atau bentuk (num_qubits, 3) masing-masing. Turutan adalah ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int atau `None` | Masa pelaksanaan maksimum pada QPU dalam saat. Jika masa runtime melebihi nilai ini, kerja akan dibatalkan. Jika `None`, had lalai yang ditetapkan oleh Qiskit Runtime akan digunakan. | `None`
`num_randomizations` | int | Bilangan pengacakan yang hendak digunakan untuk pembelajaran hingar dan twirling get. | 32
`max_layers_to_learn` | int | Bilangan maksimum lapisan unik yang hendak dipelajari. | 4
`mitigate_readout_error` | bool | Bendera Boolean yang menunjukkan sama ada hendak melakukan mitigasi ralat bacaan atau tidak. | True
`num_readout_calibration_shots` | int | Bilangan tembakan yang hendak digunakan untuk mitigasi ralat bacaan. | 10000
`default_precision` | float | Ketepatan lalai yang hendak digunakan untuk PUB yang tidak dinyatakan ketepatannya. |0.02
`seed` | int atau `None` | Tetapkan benih penjana nombor rawak untuk kebolehulangan. Jika `None`, jangan tetapkan benih. | None
## Output
[PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult) Qiskit yang mengandungi keputusan yang dikurangkan TEM. Keputusan untuk setiap PUB dikembalikan sebagai [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) yang mengandungi medan berikut:

Nama |Jenis | Keterangan
-- | -- | --
data | DataBin | [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin) Qiskit yang mengandungi pemerhatian yang dikurangkan TEM dan ralat piawainya. DataBin mempunyai medan berikut: <ul><li>`evs`: Nilai pemerhatian yang dikurangkan TEM.</li><li>`stds`: Ralat piawai pemerhatian yang dikurangkan TEM.</li></ul>
metadata | dict | Kamus yang mengandungi keputusan tambahan. Kamus mengandungi kunci berikut: <ul><li>`"evs_non_mitigated"`: Nilai pemerhatian tanpa mitigasi ralat.</li><li>`"stds_non_mitigated"`: Ralat piawai keputusan tanpa mitigasi ralat.</li><li>`"evs_mitigated_no_readout_mitigation"`: Nilai pemerhatian dengan mitigasi ralat tetapi tanpa mitigasi ralat bacaan.</li><li>`"stds_mitigated_no_readout_mitigation"`: Ralat piawai keputusan dengan mitigasi ralat tetapi tanpa mitigasi ralat bacaan.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: Nilai pemerhatian tanpa mitigasi ralat tetapi dengan mitigasi ralat bacaan.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: Ralat piawai keputusan tanpa mitigasi ralat tetapi dengan mitigasi ralat bacaan.</li></ul>
## Mendapatkan mesej ralat
Jika status beban kerja anda RALAT, gunakan job.result() untuk mendapatkan mesej ralat seperti berikut: